# Aufgabe 3: Eigenes neuronales Netz (Schachbrett)

Netz mit 5 Neuronen von Grund auf, Lernalgorithmus, Fehler-/Gewichtsverlauf.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt


## Setup (Hodgkin-Huxley-Modell und Netz-Konstanten)

Die folgenden Definitionen fassen das Hodgkin-Huxley-Modell aus Aufgabe 2 sowie die Konstanten und Hilfsfunktionen des Netzes zusammen. Sie werden von allen folgenden Aufgaben dieses Notebooks benötigt.

In [ ]:
from scipy.integrate import odeint

# ==== Setup 1: Hodgkin-Huxley-Modell (aus Aufgabe 2, hier als Baustein) ====
V_POT = -65.0; C = 1.0
U_K = -77.0; U_NA = 50.0; U_L = -54.387
G_K = 36.0; G_NA = 120.0; G_L = 0.3

def alpha_n(U): return -0.01 * (55.0 + U) / (np.exp(-(55.0 + U) / 10.0) - 1.0)
def beta_n(U):  return 0.125 * np.exp(-(65.0 + U) / 80.0)
def alpha_m(U): return -0.1 * (40.0 + U) / (np.exp(-(40.0 + U) / 10.0) - 1.0)
def beta_m(U):  return 4.0 * np.exp(-(65.0 + U) / 18.0)
def alpha_h(U): return 0.07 * np.exp(-(65.0 + U) / 20.0)
def beta_h(U):  return 1.0 / (np.exp(-(35.0 + U) / 10.0) + 1.0)

def x_infinity(alpha, beta):
    return alpha / (alpha + beta)

def ionic_currents(U, n, m, h):
    i_k = G_K * n ** 4 * (U - U_K)
    i_na = G_NA * m ** 3 * h * (U - U_NA)
    i_l = G_L * (U - U_L)
    return i_k, i_na, i_l

def rhs(state, t, I_ext):
    U, n, m, h = state
    I = I_ext(t) if callable(I_ext) else I_ext
    i_k, i_na, i_l = ionic_currents(U, n, m, h)
    dU = (I - i_k - i_na - i_l) / C
    dn = alpha_n(U) * (1 - n) - beta_n(U) * n
    dm = alpha_m(U) * (1 - m) - beta_m(U) * m
    dh = alpha_h(U) * (1 - h) - beta_h(U) * h
    return np.array([dU, dn, dm, dh])

def initial_state():
    U0 = V_POT
    n0 = x_infinity(alpha_n(U0), beta_n(U0))
    m0 = x_infinity(alpha_m(U0), beta_m(U0))
    h0 = x_infinity(alpha_h(U0), beta_h(U0))
    return np.array([U0, n0, m0, h0])

# ==== Setup 2: Netz-Konstanten und Hilfsfunktionen ====
I_0 = -5.0            # minimale/Grundstromstärke [nA]
I_MAX = 10.0          # Strom für ein aktives (schwarzes) Feld [nA]
SPIKE_SCHWELLE = 0.0  # Spannung, ab der ein Neuron feuert [mV]

def clamp_current(I):
    return np.maximum(I, I_0)

def solve_hodgkin_huxley(I_ext, t):
    y = odeint(rhs, initial_state(), t, args=(I_ext,))
    return y[:, 0]

## 3a) Aufbau überlegen (Neuronen, Verbindungen, optimale Gewichte)

Aufbau des Netzes: Für die Erkennung eines 2×2-Schachbretts werden fünf Neuronen benötigt: vier Input-Neuronen (je eines pro Feld) und ein Output-Neuron. Ein Hidden-Layer existiert nicht, die vier Inputs sind direkt mit dem Output verbunden. Es gibt also vier Verbindungen mit je einem Gewicht $w_1,\dots,w_4$.

Formen von Input und Output: Der Input ist ein Vektor aus vier Werten $[x_1,x_2,x_3,x_4]$ mit $x_i \in \{0,1\}$ (schwarz/weiß), die das 2×2-Feld zeilenweise kodieren. Ein „1"-Feld regt sein Input-Neuron mit $I_\mathrm{max}$ zum Feuern an, ein „0"-Feld hält es bei $I_0$ (Ruhe). Der Output ist ein Neuron: Feuert es, wird das Muster als Schachbrett erkannt (`True`), sonst nicht (`False`). Der Strom ins Output-Neuron ergibt sich nach Gl. (14) als gewichtete Summe der Spannungen $I_\mathrm{out} = \sum_i w_i\,U_i$.

Optimale Gewichte: Um das Zielmuster (schwarze Diagonale, z. B. $[1,0,0,1]$) von den anderen abzugrenzen, müssen die Gewichte auf den beiden schwarzen Diagonalfeldern hoch und auf den übrigen null sein, also z. B. $w = [1,0,0,1]$. Dann treiben nur die zwei gemeinsam feuernden Diagonal-Neuronen das Output über die Schwelle, während einzelne Felder oder die andere Diagonale es nicht schaffen. Die absolute Größe der Gewichte ist dabei nebensächlich, entscheidend ist das Verhältnis (Diagonale stark, Rest schwach); Skalierungen wie $[0.5,0,0,0.5]$ oder $[2,0,0,2]$ liefern dasselbe Ergebnis.

Wieso nur ein Diagonalmuster erkannt werden kann: Die Wichtungsfaktoren haben die Einheit einer elektrischen Leitfähigkeit und sind daher physikalisch positiv ($w_i \ge 0$). Bei ausschließlich positiven Gewichten kann ein zusätzliches aktives (schwarzes) Feld den Strom ins Output-Neuron nur erhöhen, nie senken:
$$I_\mathrm{out}(1,1,1,1) = I_\mathrm{out}(1,0,0,1) + \underbrace{w_2\,U_2^{\text{feuernd}} + w_3\,U_3^{\text{feuernd}}}_{\ge 0}\,.$$
Der Antrieb von $[1,1,1,1]$ ist also stets $\ge$ dem von $[1,0,0,1]$. Feuert das Zielmuster, muss $[1,1,1,1]$ erst recht feuern. Das Netz kann die Diagonalfelder belohnen, die übrigen Felder aber nicht bestrafen, dafür bräuchte es negative Gewichte (eine „Gegenstimme"), die als negative Leitfähigkeit physikalisch nicht existieren.
Die Verwendung negativer Gewichte entgegen der physikalischen Realität führen zu falschen Ergebnissen, da diese mit negativen Spannungen Muster als falsch positiv erkennt. 

Konsequenz: Das Netz erkennt zuverlässig „schwarze Diagonale", kann aber die drei Muster $[1,0,1,1]$, $[1,1,0,1]$ und $[1,1,1,1]$ (Diagonale schwarz plus Extra-Felder) nicht ausschließen. Damit ist das erreichbare Optimum 13 von 16 korrekt, keine Schwäche der Implementierung, sondern die prinzipielle Grenze eines einlagigen Netzes mit positiven Gewichten. Ein echtes Schachbrett (Diagonale schwarz und die anderen Felder weiß) sowie beide Schachbretter zugleich erfordern eine versteckte Schicht.

## 3b) NN mit 5 Neuronen (I_0 als Minimum)

In dieser Zelle wird die Funktion predict definiert und an zwei Beispielmustern getestet. predict setzt das 2x2 Muster in Eingangsströme um (schwarzes Feld auf I_MAX, weißes Feld auf I_0), löst für jedes der vier Input-Neuronen und für das eine Output-Neuron das Hodgkin-Huxley-Modell und prüft, ob das Output-Neuron feuert. Über die Funktion clamp_current wird sichergestellt, dass der Strom in einem Neuron den Minimalwert I_0 nicht unterschreitet, wie in der Aufgabe gefordert.

Das Ergebnis bestätigt, dass das Netz aus fünf Neuronen technisch funktioniert. Das Zielmuster (1,0,0,1) löst ein Feuern des Output-Neurons aus (Ausgabe True), während das leere Feld (0,0,0,0) kein Feuern erzeugt (Ausgabe False). Damit ist gezeigt, dass die Grundmechanik richtig arbeitet. Ob das Netz auch zuverlässig zwischen Schachbrett und Nichtschachbrett trennt, wird erst in Aufgabe 3c über alle 16 Muster geprüft.

In [ ]:
# 3b) predict: erkennt, ob ein 2x2-Muster ein Schachbrett ist
def predict(weights, pattern, t=None):
    if t is None:
        t = np.arange(0, 50, 0.01)

    # 1. Eingangsströme der 4 Input-Neuronen festlegen
    I_in = []
    for i in range(4):
        if pattern[i] == 1:
            I_in.append(I_MAX)
        else:
            I_in.append(I_0)

    # 2. Für jedes Input-Neuron das HHM lösen und die Spannung merken
    U = []
    for i in range(4):
        U.append(solve_hodgkin_huxley(I_in[i], t))
    U = np.array(U)

    # 3. Strom ins Output-Neuron nach Gleichung (14): Summe w_i * U_i
    I_out = np.zeros(len(t))
    for i in range(4):
        I_out = I_out + weights[i] * U[i]
    I_out = clamp_current(I_out)   # Minimum I_0 nicht unterschreiten

    # 4. Output-Neuron mit diesem zeitabhängigen Strom lösen
    def I_out_funktion(zeit):
        return np.interp(zeit, t, I_out)
    U_out = solve_hodgkin_huxley(I_out_funktion, t)

    # 5. Prüfen, ob das Output-Neuron gefeuert hat
    hat_gefeuert = False
    for wert in U_out:
        if wert > SPIKE_SCHWELLE:
            hat_gefeuert = True
    return hat_gefeuert


# Schnelltest: läuft predict und liefert True/False?
w = np.array([1.0, 0.0, 0.0, 1.0])
print("Zielmuster [1,0,0,1]:", predict(w, np.array([1, 0, 0, 1])))
print("Leeres Feld [0,0,0,0]:", predict(w, np.array([0, 0, 0, 0])))

## 3c) Alle Gewichte = 1 vs. optimale Gewichte

In dieser Zelle wird über alle 16 möglichen Muster geprüft, wie gut das Netz mit einem bestimmten Gewichtssatz zwischen Schachbrett und Nichtschachbrett unterscheidet. Die Funktion teste_gewichte ruft für jedes Muster predict auf, vergleicht die Ausgabe mit dem Sollwert (nur (1,0,0,1) gilt als Schachbrett) und zählt die korrekten Treffer. Geprüft werden zwei Gewichtssätze: alle Gewichte gleich 1 und der optimale positive Satz mit hohen Gewichten auf der Diagonale.

Mit gleichen Gewichten erkennt das Netz das Schachbrett nicht zuverlässig. Da alle Felder gleich stark eingehen, feuert das Output-Neuron vor allem nach der Anzahl aktiver Felder und trennt die Klassen kaum. Mit dem optimalen positiven Satz (1,0,0,1) steigt die Zahl der korrekten Muster deutlich auf 13 von 16. Die verbleibenden drei Fehler treten immer bei den Mustern (1,0,1,1), (1,1,0,1) und (1,1,1,1) auf, also genau dann, wenn die Zieldiagonale schwarz ist und zusätzlich weitere Felder aktiv sind. Die Antwort auf die Frage, ob das Ergebnis immer stimmt, lautet somit nein. Auch mit optimalen Gewichten bleiben diese drei Muster falsch, weil positive Gewichte die anderen Felder nicht bestrafen können. Das ist die in Aufgabe 3a beschriebene prinzipielle Grenze und motiviert den Lernalgorithmus in 3d.

In [ ]:
import itertools

# alle 16 möglichen 2x2-Muster
muster = list(itertools.product([0, 1], repeat=4))

def ist_schachbrett(p):
    return p == (1, 0, 0, 1)      # dein Zielmuster (ggf. anpassen)

def teste_gewichte(weights):
    weights = np.array(weights, dtype=float)
    korrekt = 0
    for p in muster:
        vorhersage = predict(weights, np.array(p))
        soll = ist_schachbrett(p)
        if vorhersage == soll:
            korrekt += 1
            markierung = ""
        else:
            markierung = "   <-- falsch"
        print(f"{p}   Vorhersage: {str(vorhersage):5}   Soll: {str(soll):5}{markierung}")
    print(f"\n{korrekt} von {len(muster)} korrekt\n")

# 3c, Teil 1: alle Gewichte = 1
print("=== Alle Gewichte = 1 ===")
teste_gewichte([1, 1, 1, 1])

# 3c, Teil 2: optimale Gewichte
print("=== Optimale Gewichte ===")
teste_gewichte([1, 0, 0, 1])

## 3d) Maschinelles Lernen (Gewichte je Prüfergebnis anpassen)

In dieser Zelle wird der Lernalgorithmus (Funktion train) definiert und ausgeführt. Zunächst werden alle 16 Muster erzeugt und das Zielmuster mehrfach hinzugefügt, damit die seltene Schachbrettklasse im Training ausreichend oft vorkommt (Balancing). Die Funktion train startet mit zufälligen positiven Gewichten aus dem Bereich (0,1] und geht die Muster mehrere Durchgänge lang durch. Für jedes Muster vergleicht sie die Vorhersage mit dem Sollwert und passt die Gewichte der aktiven Felder in Richtung des Fehlers an. Sollte das Netz feuern, tat es aber nicht, werden die Gewichte erhöht, im umgekehrten Fall gesenkt. Nach jeder Anpassung werden die Gewichte auf Werte größer oder gleich null geklemmt, damit sie physikalisch gültige Leitfähigkeiten bleiben. Fehlerzahl und Gewichte werden je Durchgang protokolliert.

Der gelernte Gewichtssatz läuft von selbst auf die in 3a hergeleitete Struktur zu, also hohe Gewichte auf den beiden Diagonalfeldern und Werte nahe null auf den übrigen. Das Netz findet damit eigenständig die beste positive Lösung, ohne dass die optimalen Gewichte vorgegeben werden. Zum Abschluss wird die Leistung der gelernten Gewichte mit teste_gewichte über alle 16 Muster überprüft.

Beobachtung zur Laufzeit: Der Algorithmus konvergiert bei diesem kleinen Problem sehr schnell, meist bleiben Fehler und Gewichte schon nach etwa zwei Durchgängen konstant. Die hier gewählten 25 Epochen sind also ein sicherer Überschätzer; für kürzere Rechenzeit genügt epochs=5. Das Training darf zudem ein grobes Zeitgitter verwenden, da es die richtige Gewichtsstruktur auch damit findet, während die abschließende Prüfung über teste_gewichte mit dem feineren Standardgitter läuft.

In [ ]:
# 3d) Lernalgorithmus (Perzeptron-Regel)
def train(patterns, targets, learning_rate=0.1, epochs=100, seed=None, t=None):
    # Gröberes Zeitgitter fürs Training -> viel schneller (predict wird sehr oft aufgerufen)
    if t is None:
        t = np.arange(0, 50, 0.05)
    # Startgewichte zufällig in (0, 1], positiv (Leitfähigkeit)
    rng = np.random.default_rng(seed)
    weights = rng.uniform(0.0, 1.0, size=4)

    patterns = [np.array(p) for p in patterns]
    fehler_verlauf = []
    gewichte_verlauf = []
    for epoch in range(epochs):
        fehler_summe = 0
        for p, ziel in zip(patterns, targets):
            vorhersage = int(predict(weights, p, t=t))
            fehler = ziel - vorhersage
            if fehler != 0:
                weights = weights + learning_rate * fehler * p
                weights = np.maximum(weights, 0.0)   # Leitfähigkeit >= 0
                fehler_summe += 1
        fehler_verlauf.append(fehler_summe)
        gewichte_verlauf.append(weights.copy())
    return weights, np.array(fehler_verlauf), np.array(gewichte_verlauf)


pats = list(itertools.product([0, 1], repeat=4))
ziel = (1, 0, 0, 1)

# Balancing: Zielmuster mehrfach zeigen, damit das Netz nicht "immer nein" lernt
patterns = pats + [ziel] * 6
targets  = [1 if p == ziel else 0 for p in pats] + [1] * 6

# Achtung: Training ruft predict sehr oft auf -> dauert einige Minuten.
# Für schnellere Läufe epochs verringern oder gröberes t übergeben.
w, fehler_verlauf, gewichte_verlauf = train(
    patterns, targets, learning_rate=0.2, epochs=25, seed=1)

print("Gelernte Gewichte:", np.round(w, 2))
print("Fehler pro Durchgang:", list(fehler_verlauf))

# Performance der gelernten Gewichte über alle 16 Muster prüfen
print("\n=== Performance der gelernten Gewichte ===")
teste_gewichte(w)

## 3e) Fehler und Gewichte über Durchgänge grafisch

In dieser Zelle werden die im Training protokollierten Größen grafisch dargestellt. Der linke Plot zeigt die Anzahl der Fehler pro Durchgang, der rechte die Entwicklung der vier Gewichte über die Durchgänge.

Der Fehlerverlauf sinkt in den ersten Durchgängen und stabilisiert sich anschließend bei einem kleinen Wert größer als null. Das Netz wird also nicht vollständig fehlerfrei, sondern erreicht sein Optimum von 13 von 16 korrekt erkannten Mustern. Das deckt sich mit der Grenze aus Aufgabe 3a, denn die drei Muster mit schwarzer Diagonale und zusätzlichen aktiven Feldern lassen sich mit positiven Gewichten nicht abtrennen. Der Gewichtsverlauf zeigt, wie die beiden Diagonalgewichte hoch bleiben, während die anderen beiden gegen null laufen. Wiederholt man den Lernvorgang mit verschiedenen Startwerten, landet er reproduzierbar bei derselben Struktur, was für die Robustheit des Algorithmus spricht.

Bemerkenswert ist, dass die Gewichte ab dem Optimum vollständig konstant bleiben. Die verbliebenen Fehler heben sich gegenseitig auf: Ein falsch positives und das zugehörige falsch negative Muster ziehen dieselben Gewichte um denselben Betrag in entgegengesetzte Richtungen, sodass sich die Änderungen über einen Durchgang genau kompensieren. Das Netz sitzt damit in einem stabilen Gleichgewicht. Eine feinere Zeitauflösung verschiebt die prinzipielle Grenze von 13 von 16 nicht, beseitigt aber Artefakte, bei denen ein zu grobes Gitter einzelne Muster falsch einordnet.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Fehler ueber die Durchgaenge
ax1.plot(fehler_verlauf, marker="o")
ax1.set_xlabel("Durchgang (Epoche)")
ax1.set_ylabel("Anzahl Fehler")
ax1.set_title("Fehler ueber die Durchgaenge")
ax1.grid(alpha=0.3)

# Entwicklung der vier Gewichte
for i in range(4):
    ax2.plot(gewichte_verlauf[:, i], label=f"w{i+1}")
ax2.set_xlabel("Durchgang (Epoche)")
ax2.set_ylabel("Gewicht")
ax2.set_title("Entwicklung der Gewichte")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Robustheit (3e): mehrmals mit verschiedenen seeds trainieren und vergleichen.
# Achtung: dauert entsprechend laenger.
# for s in range(3):
#     w_s, f_s, _ = train(patterns, targets, learning_rate=0.2, epochs=25, seed=s)
#     print(f"seed {s}: Endfehler {f_s[-1]}, Gewichte {np.round(w_s,2)}")